# Derive CHM from DSM

##  Imports

In [1]:
import os
from pathlib import Path
import glob
import xarray as xr
import rioxarray as rxr
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import minimum_filter, maximum_filter, gaussian_filter

## Define File Paths

In [2]:
notebook_dir = Path.cwd()
base_path = notebook_dir.parent / "Data" 
base_path_str = str(base_path)
print(f"Base path automatically set to: {base_path_str}")

Base path automatically set to: /net/home/sloeblein/Patras_Wildfire/Preprocessing/Data


In [ ]:
folder_path = os.path.join(base_path, "4_Resampled_Grids")
all_files = glob.glob(os.path.join(folder_path, "*.tif"))
output_folder = os.path.join(base_path, "5_Added_CHM_Channel"
os.makedirs(output_folder, exist_ok=True)

In [ ]:
NODATA_VAL = -9999.0

# Loop through all available files
for file in all_files:
    basename = os.path.basename(file)
    print(f"Creating CHM from DSM {basename}...")

    # 1. Load image
    ds = rxr.open_rasterio(file)

    # 2. Select DSM (Band 4)
    dsm = ds.sel(band=4)

    # --- EDGE FIX: Prevent artificial walls at the borders ---
    # Convert the black border (exactly 0) into NaN values.
    # This forces the morphological filters to ignore the border area.
    dsm_valid = dsm.where(dsm != NODATA_VAL)

    # 3. DEFINE FILTER PARAMETERS
    # Resolution setting: 1m = 40 pixels (at 2.5cm/pixel resolution).
    # We choose 8 meters (320 pixels) to be wider than the largest tree crown.
    window_size_pixels = 320 
    print(f"Calculating DTM via sliding window ({window_size_pixels} px / approx. 8m)...")

    # 4. SLIDING WINDOW PROCESSING (Morphological Opening)
    # IMPORTANT: We replace the NaN borders with an extremely high value (+99999.0).
    # If we left them as -9999.0, the minimum filter would drag the black border 
    # hundreds of pixels deep into the actual forest!
    dsm_filled = np.nan_to_num(dsm_valid.values, nan=9999.0)

    # a) Minimum filter (Erosion): Removes trees but pulls slopes into terraces/steps
    print(" -> Step 1: Running Minimum filter...")
    dtm_min = minimum_filter(dsm_filled, size=window_size_pixels)

    # b) Maximum filter (Dilation): Fixes the artificial terraces on the slopes
    print(" -> Step 2: Running Maximum filter (Slope correction)...")
    dtm_raw = maximum_filter(dtm_min, size=window_size_pixels)

    # c) Gaussian filter: Smooths the ground surface for a natural look
    print(" -> Step 3: Running Gaussian filter (Smoothing)...")
    dtm_smooth = gaussian_filter(dtm_raw, sigma=window_size_pixels/4)

    # 5. PACK BACK INTO XARRAY DATAARRAY
    dtm = xr.DataArray(
        dtm_smooth, 
        coords=dsm.coords, 
        dims=dsm.dims
    )

    # 6. CALCULATE CANOPY HEIGHT MODEL (CHM)
    print(" -> Step 4: Calculating final CHM...")
    chm = dsm_valid - dtm

    # Post-processing cleanup: enforce non-negative heights 
    chm = chm.where(chm > 0, 0)
    
    # Get Nodata value from other bands
    target_nodata = ds.sel(band=1).rio.nodata
    
    # If channel 1 "None" as NodataValue, then set it to np.nan
    if target_nodata is None:
        target_nodata = np.nan

    # Set all nans to target value
    chm = chm.fillna(target_nodata)

    # Optional safety cutoff to catch edge interpolation artifacts
    # chm = chm.where(chm < 20, 0) # Ignores everything unrealistically taller than 20 meters

    # RESTORE STRICT NODATA: Fill the NaNs back up with exactly -9999.0!
    chm = chm.fillna(NODATA_VAL)
    
    # Ensure rioxarray knows about the NoData value before saving
    chm.rio.write_nodata(NODATA_VAL, inplace=True)

    # 7. Plot for visual control
    plt.clf()
    chm.where(chm != NODATA_VAL).plot(cmap="viridis") # or gray
    plt.title(f"CHM (Morphological Filter) - {basename}")
    plt.show()

    # 8. Save updated dataset to the output folder
    ds_out = ds.copy()
    ds_out.loc[dict(band=4)] = chm.values

    # Make absolutely sure the global dataset metadata knows about -9999.0
    ds_out.rio.write_nodata(NODATA_VAL, inplace=True)
    
    output_path = os.path.join(output_folder, basename)
    ds_out.rio.to_raster(output_path)
    
    print(f" -> Saved: {output_path}")
    
    # Close datasets to free up memory
    ds.close()
    ds_out.close()
    print("\n✅ DONE!")